In [5]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [6]:
import sys
!{sys.executable} -m pip install --user python-woc

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'knygren/glmbayes',
    'rapidsai/cuxfilter',
    'NFFT/nfft',
    'cselab/aphros',
    'learning-at-home/hivemind',
    'epics-base/pva2pva',
    'GEOS-ESM/MAPL',
    'icbi-lab/luca',
    'ajdawson/windspharm',
    'sonvt1710/deepface'
]

list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00072e026eadd6ffc3482d026f501f92577bbcd1,knygren_glmbayes


In [10]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
# chunks = [df['sha1'][x:x+40] for x in range(0, len(df), 40)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(2)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 10%|███████▉                                                                     | 440/4302 [14:54<2:11:00,  2.04s/it]

Got Errors {'47ec5476d059b51f1a7ebf382adb248b27ccae2f': 'Key 47ec5476d059b51f1a7ebf382adb248b27ccae2f not found in /da5_fast/All.sha1c/commit_71.tch'}


 12%|█████████▎                                                                   | 520/4302 [17:36<2:08:03,  2.03s/it]

Got Errors {'7d5364979b2115edd917ba738038aa940836e734': 'Key 7d5364979b2115edd917ba738038aa940836e734 not found in /da5_fast/All.sha1c/commit_125.tch'}


 12%|█████████▍                                                                   | 526/4302 [17:49<2:08:23,  2.04s/it]

...

Got Errors {'e386deec8f13e16a3ba9f8e38f920ed6d80e5b63': 'Key e386deec8f13e16a3ba9f8e38f920ed6d80e5b63 not found in /da5_fast/All.sha1c/commit_99.tch'}


 95%|████████████████████████████████████████████████████████████████████████▎   | 4093/4302 [2:19:05<07:05,  2.04s/it]

Got Errors {'f89f96a7fe1ffd7f36e652598511fbf70bb74e0c': 'Key f89f96a7fe1ffd7f36e652598511fbf70bb74e0c not found in /da5_fast/All.sha1c/commit_120.tch'}


100%|████████████████████████████████████████████████████████████████████████████| 4302/4302 [2:26:13<00:00,  2.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00072e026eadd6ffc3482d026f501f92577bbcd1,119a970aec45c0d7fabb05ae37e57bb9a18f13e4,[50bb507827200f3ca3dba617e77e6c8757bd9baa],Nygren <Kjell.Nygren@sanofi.com>,1596427257,-0400,Nygren <Kjell.Nygren@sanofi.com>,1596427257,-0400,Finished most cleanup for the new rIndependent...,00072e026eadd6ffc3482d026f501f92577bbcd1,knygren_glmbayes
1,00489b17e85cab12b6a3624ed9ac3df8c202e1e1,982d4e24713fa5efd963a6607fb1aacb32f5af62,[28062eff089c040a4bf4e21dcd00bf2fc689063e],knygren <kjell.a.nygren@gmail.com>,1752529616,-0400,knygren <kjell.a.nygren@gmail.com>,1752529640,-0400,Add Initial News.MD for version tracking\n,00489b17e85cab12b6a3624ed9ac3df8c202e1e1,knygren_glmbayes


In [4]:
print("Total commits:", len(df))
print("Total chunks:", len(chunks))

Total commits: 43011
Total chunks: 4302


In [5]:
df.groupby("project").size().sort_values(ascending=False)

project
geos-esm_mapl                14978
cselab_aphros                10702
learning-at-home_hivemind     6394
nfft_nfft                     3973
rapidsai_cuxfilter            2688
sonvt1710_deepface            2071
epics-base_pva2pva             665
knygren_glmbayes               581
icbi-lab_luca                  578
ajdawson_windspharm            381
dtype: int64

In [8]:
test_chunk = chunks[0].to_list()

res, err = woc.get_values_many('commit.tch', test_chunk)

print("Requested:", len(test_chunk))
print("Received:", len(res))
print("Errors:", err)

Requested: 40
Received: 40
Errors: {}


In [11]:
print("Expected commits:", len(df))
print("Retrieved commits:", len(df_commit_data))
print("Missing commits:", len(df) - len(df_commit_data))

df_commit_data.groupby("project").size().sort_values(ascending=False)

Expected commits: 43011
Retrieved commits: 42980
Missing commits: 31


project
geos-esm_mapl                14975
cselab_aphros                10702
learning-at-home_hivemind     6394
nfft_nfft                     3965
rapidsai_cuxfilter            2688
sonvt1710_deepface            2071
epics-base_pva2pva             660
knygren_glmbayes               581
icbi-lab_luca                  578
ajdawson_windspharm            366
dtype: int64

In [17]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00072e026eadd6ffc3482d026f501f92577bbcd1', 'tree': '119a970aec45c0d7fabb05ae37e57bb9a18f13e4', 'parent': ['50bb507827200f3ca3dba617e77e6c8757bd9baa'], 'author': 'Nygren <Kjell.Nygren@sanofi.com>', 'author_time': 1596427257, 'author_tz': '-0400', 'committer': 'Nygren <Kjell.Nygren@sanofi.com>', 'committer_time': 1596427257, 'committer_tz': '-0400', 'message': 'Finished most cleanup for the new rIndependent_Normal_Gamma_reg function\n'}


In [13]:
# # Now that the data is retrieved, save it and commit to your GH fork
# # First, we need to flatten info in order to export as csv
# # our dataframe will have columns project,'commit, author, time, message'
# dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
# for k in commit_data:
#   row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
#   dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [20]:
dfinf = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

dfinf = dfinf.rename(columns={
    'project': 'project_wocid',
    'commit': 'commit_sha1',
    'author_time': 'time',
    'message': 'commit message'
})

dfinf.head()

,project_wocid,commit_sha1,author,time,commit message
0,knygren_glmbayes,00072e026eadd6ffc3482d026f501f92577bbcd1,Nygren <Kjell.Nygren@sanofi.com>,1596427257,Finished most cleanup for the new rIndependent...
1,knygren_glmbayes,00489b17e85cab12b6a3624ed9ac3df8c202e1e1,knygren <kjell.a.nygren@gmail.com>,1752529616,Add Initial News.MD for version tracking\n
2,knygren_glmbayes,0096d32f114cc9233dd817f1df47343073340294,knygren <kjell.a.nygren@gmail.com>,1757498572,Adding Rdpack capabilities to ensure enabled b...
3,knygren_glmbayes,0102d1080849964f0ee67b9815bffbef1ad34b02,knygren <kjell.a.nygren@gmail.com>,1753958575,Adds PKG_CPP_FLAGS to configure script output.\n
4,knygren_glmbayes,0269eb3d3172fe0038f54f249148737b2c75ecb2,knygren <kjell.a.nygren@gmail.com>,1753981278,Additional edits to configure script\n


In [18]:
# check if it has the right content
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,knygren_glmbayes,00072e026eadd6ffc3482d026f501f92577bbcd1,Nygren <Kjell.Nygren@sanofi.com>,1596427257,Finished most cleanup for the new rIndependent...


In [21]:
#mode a means append, so you have all your projects in the same file
yournetid='aalkafou'
# dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

dfinf.to_csv(
    yournetid + '_project_summary.csv',
    index=False,
    sep=';'
)

In [22]:
check = pd.read_csv('aalkafou_project_summary.csv', sep=';')

print("Rows:", len(check))
print("Columns:", check.columns.tolist())
print("\nCommits per project:")
print(check['project_wocid'].value_counts())

Rows: 42980
Columns: ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']

Commits per project:
project_wocid
geos-esm_mapl                14975
cselab_aphros                10702
learning-at-home_hivemind     6394
nfft_nfft                     3965
rapidsai_cuxfilter            2688
sonvt1710_deepface            2071
epics-base_pva2pva             660
knygren_glmbayes               581
icbi-lab_luca                  578
ajdawson_windspharm            366
Name: count, dtype: int64


In [23]:
summary = df_commit_data.groupby('project').agg(
    num_commits=('commit', 'count'),
    num_authors=('author', 'nunique'),
    min_time=('author_time', 'min'),
    max_time=('author_time', 'max')
)

summary

,num_commits,num_authors,min_time,max_time
project,,,,
ajdawson_windspharm,366,23,1339869972,1759355002
cselab_aphros,10702,8,1511875620,1753041425
epics-base_pva2pva,660,19,1441393734,1753693649
geos-esm_mapl,14975,99,1559916412,1776373147
icbi-lab_luca,578,8,1603960891,1737283318
knygren_glmbayes,581,7,1396699454,1762169096
learning-at-home_hivemind,6394,89,1583097061,1760425343
nfft_nfft,3965,55,1113384599,1761988874
rapidsai_cuxfilter,2688,76,1529012860,1762365123


In [24]:
summary_readable = summary.copy()

summary_readable['min_time'] = pd.to_datetime(
    summary_readable['min_time'], unit='s'
).dt.strftime('%Y-%m-%d')

summary_readable['max_time'] = pd.to_datetime(
    summary_readable['max_time'], unit='s'
).dt.strftime('%Y-%m-%d')

summary_readable

,num_commits,num_authors,min_time,max_time
project,,,,
ajdawson_windspharm,366,23,2012-06-16,2025-10-01
cselab_aphros,10702,8,2017-11-28,2025-07-20
epics-base_pva2pva,660,19,2015-09-04,2025-07-28
geos-esm_mapl,14975,99,2019-06-07,2026-04-16
icbi-lab_luca,578,8,2020-10-29,2025-01-19
knygren_glmbayes,581,7,2014-04-05,2025-11-03
learning-at-home_hivemind,6394,89,2020-03-01,2025-10-14
nfft_nfft,3965,55,2005-04-13,2025-11-01
rapidsai_cuxfilter,2688,76,2018-06-14,2025-11-05


In [25]:
import requests

github_repos = [
    'knygren/glmbayes',
    'rapidsai/cuxfilter',
    'NFFT/nfft',
    'cselab/aphros',
    'learning-at-home/hivemind',
    'epics-base/pva2pva',
    'GEOS-ESM/MAPL',
    'icbi-lab/luca',
    'ajdawson/windspharm',
    'sonvt1710/deepface'
]

for repo in github_repos:
    info = requests.get(f'https://api.github.com/repos/{repo}').json()
    commits = requests.get(f'https://api.github.com/repos/{repo}/commits').json()

    print(repo)
    print("Stars:", info.get('stargazers_count'))
    print("Forks:", info.get('forks_count'))
    print("Last commit:", commits[0]['commit']['committer']['date'][:10])
    print()

knygren/glmbayes
Stars: 5
Forks: 1
Last commit: 2026-09-27

rapidsai/cuxfilter
Stars: 313
Forks: 81
Last commit: 2026-06-22

NFFT/nfft
Stars: 188
Forks: 51
Last commit: 2026-09-28

cselab/aphros
Stars: 460
Forks: 52
Last commit: 2025-07-20

learning-at-home/hivemind
Stars: 2529
Forks: 233
Last commit: 2026-01-04

epics-base/pva2pva
Stars: 5
Forks: 15
Last commit: 2026-08-02

GEOS-ESM/MAPL
Stars: 45
Forks: 25
Last commit: 2026-09-23

icbi-lab/luca
Stars: 94
Forks: 30
Last commit: 2025-01-19

ajdawson/windspharm
Stars: 100
Forks: 39
Last commit: 2026-08-25

sonvt1710/deepface
Stars: 0
Forks: 0
Last commit: 2026-09-28



# Make sure you check in to your fork not just the notebook but also the csv files!!!

## Part 1 — Project Summary

The following summarizes the GitHub repository information and World of Code (WoC) commit data for the 10 assigned projects.
    

| Project | WoC Commits | Authors | Min Time | Max Time | GitHub Stars | GitHub Forks | Last GitHub Commit |
|---|---:|---:|---|---|---:|---:|---|
| ajdawson_windspharm | 366 | 23 | 2012-06-16 | 2025-10-01 | 100 | 39 | 2026-08-25 |
| cselab_aphros | 10,702 | 8 | 2017-11-28 | 2025-07-20 | 460 | 52 | 2025-07-20 |
| epics-base_pva2pva | 660 | 19 | 2015-09-04 | 2025-07-28 | 5 | 15 | 2026-08-02 |
| geos-esm_mapl | 14,975 | 99 | 2019-06-07 | 2026-04-16 | 45 | 25 | 2026-09-23 |
| icbi-lab_luca | 578 | 8 | 2020-10-29 | 2025-01-19 | 94 | 30 | 2025-01-19 |
| knygren_glmbayes | 581 | 7 | 2014-04-05 | 2025-11-03 | 5 | 1 | 2026-09-27 |
| learning-at-home_hivemind | 6,394 | 89 | 2020-03-01 | 2025-10-14 | 2,529 | 233 | 2026-01-04 |
| nfft_nfft | 3,965 | 55 | 2005-04-13 | 2025-11-01 | 188 | 51 | 2026-09-28 |
| rapidsai_cuxfilter | 2,688 | 76 | 2018-06-14 | 2025-11-05 | 313 | 81 | 2026-06-22 |
| sonvt1710_deepface | 2,071 | 110 | 2020-02-08 | 2025-10-21 | 0 | 0 | 2026-09-28 |

In [28]:
print("CSV rows:", len(check))
print("Number of projects:", check['project_wocid'].nunique())
print("Duplicate commit SHA1s:", check['commit_sha1'].duplicated().sum())
print("Missing project:", check['project_wocid'].isna().sum())
print("Missing SHA1:", check['commit_sha1'].isna().sum())
print("Missing author:", check['author'].isna().sum())
print("Missing time:", check['time'].isna().sum())
print("Missing message:", check['commit message'].isna().sum())

CSV rows: 42980
Number of projects: 10
Duplicate commit SHA1s: 0
Missing project: 0
Missing SHA1: 0
Missing author: 0
Missing time: 0
Missing message: 2


# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github